# 第2章：栈的表达式求值实验 — 章节介绍

## 1. 前置要求

学习本章节之前，请确保你已具备以下能力或已完成相关学习：

- **前置课程**：已完成第2章（并行计算）的学习，了解 Ascend C 算子的基本开发流程。
- **知识储备**：
  - 熟悉 C/C++ 基础语法（指针、数组、结构体）。
  - 了解栈（Stack）的基本概念：LIFO 特性、Push/Pop 操作。
  - 了解 Ascend C 的 Kernel 类、GlobalTensor、`__aicore__` 关键字等基础概念。
- **环境要求**：CANN 9.0.0+，昇腾 910B3 NPU 环境（或 CANNLab 云开发环境）。

## 2. 章节目标

完成本章学习后，你将能够：

1. 理解栈的 LIFO 特性在 NPU Local Memory 上的直接实现方式。
2. 掌握 Kernel 类成员数组 + top 指针模拟顺序栈的方法。
3. 理解括号匹配的三种失配场景及其在 Ascend C 中的对应处理。
4. 理解后缀表达式求值的栈操作过程。
5. 理解中缀转后缀的"格式转换"思想及其与 Im2Col 的类比。
6. 独立完成算子编译、部署和精度验证的完整流程。

## 3. 章节内容

本章包含以下小节：

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>小节</th>
      <th>内容</th>
      <th>链接</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>02.01 章节介绍</td>
      <td>实验背景、栈与 Ascend C 的映射、三个算子概览、学习目标</td>
      <td>当前页面</td>
    </tr>
    <tr>
      <td>02.02 动手实验</td>
      <td>编译3个算子、生成测试数据、运行 Benchmark 验证精度</td>
      <td><a href="./02.02_stack_expr_lab.ipynb">02.02_stack_expr_lab.ipynb</a></td>
    </tr>
    <tr>
      <td>02.03 课后测试</td>
      <td>选择题、填空题，检验栈结构与算子开发的理解</td>
      <td><a href="./02.03_chapter_test.ipynb">02.03_chapter_test.ipynb</a></td>
    </tr>
  </tbody>
</table>

## 4. 实验背景

**栈（Stack）** 是限定仅在表尾进行插入和删除操作的线性表，其核心特征为 **后进先出（LIFO）**。

表达式求值是栈最经典的应用场景：
- **括号匹配**：利用栈检查括号是否正确配对（§3.4.5）
- **后缀表达式求值**：操作数入栈，运算符出栈计算后结果再入栈（§3.4.2）
- **中缀转后缀**：运算符栈辅助完成格式转换（§3.4.3）

本实验将在华为昇腾平台上，用 Ascend C 编程框架在 NPU 的 **Local Memory** 上直接实现栈结构，完成这3个算子。

## 5. 栈与 Ascend C 的映射

左侧为 CPU 侧顺序栈（SqStack），右侧为昇腾 NPU 侧 Kernel 类实现（成员数组即 Local Memory 栈空间），两者一一映射：

```
      CPU side                     NPU side            
+------------------+       +--------------------------+
| SqStack          |       | Kernel Class             |
|  .elem -> array  |  <->  |  char stack[N];          |
|  .top -> index   |       |  uint32_t top;           |
|  .stacksize      |       |  // Local Memory         |
+------------------+       +--------------------------+

Push: S.elem[++top] = e   ->   stack[top++] = ch
Pop:  e = S.elem[top--]   ->   ch = stack[--top]
```

**关键认知**：
- **Local Memory** = NPU 片内高速存储 ≈ 顺序栈的数组
- Kernel 类的成员数组 = 栈的存储空间（编译器自动分配到 Local Memory）
- top 指针 = 栈顶位置标记
- 标量操作（逐字符处理）适合 AI Core 的 Scalar 单元

> **注意**：在 CANN 9.0+ 中，`__ubuf__` 关键字仅用于指针类型转换，不能用于数组声明。
> Kernel 类成员数组默认分配到 Local Memory，无需额外关键字。

## 6. 三个算子概览

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr><th>算子</th><th>输入</th><th>输出</th><th>核心算法</th></tr>
  </thead>
  <tbody>
    <tr><td>BracketMatchLite</td><td>字符数组（表达式）</td><td>int32（0=正确,1/2/3=失配类型）</td><td>左括号Push，右括号Pop配对</td></tr>
    <tr><td>SuffixEvalLite</td><td>int32数组（后缀表达式token序列）</td><td>float（计算结果）</td><td>操作数Push，运算符Pop计算</td></tr>
    <tr><td>InfixToPostfixLite</td><td>字符数组（中缀表达式）</td><td>字符数组（后缀表达式）</td><td>运算符栈辅助转换</td></tr>
  </tbody>
</table>

**多核并行策略（SPMD）**：

```
输入: N个独立表达式
       ↓ 按核数切分（blockLength = totalLen / blockDim）
Core0: 表达式[0..K-1]
Core1: 表达式[K..2K-1]
...
CoreN: 表达式[...]
       ↓ 各核独立求值（核间无通信、无同步）
输出: N个结果
```

## 7. 拓扑图

<div style="text-align: left;">
  <img src="./images/stack_pipeline.svg" alt="栈表达式求值实验拓扑图" width="720">
</div>

**下一步**：打开 [02.02_stack_expr_lab.ipynb](./02.02_stack_expr_lab.ipynb) 开始动手实验！